In [ ]:
# ============================================================
# SYNTACTIC AND SEMANTIC TEXT ANALYSIS
# Materi:
# 1. Grammar Parsing with NLTK
# 2. Dependency Parsing with SpaCy
# 3. Named Entity Recognition (NER) with SpaCy
# ============================================================

# Install jika belum:
# pip install nltk spacy pandas
# python -m spacy download en_core_web_sm

import nltk
import spacy
import pandas as pd

from nltk import CFG
from nltk.parse import ChartParser

# ============================================================
# 0. SETUP
# ============================================================

# Download tokenizer NLTK.
# Digunakan jika nanti ingin tokenisasi dengan NLTK.
nltk.download("punkt")

# Load model bahasa Inggris dari SpaCy.
# en_core_web_sm sudah cukup untuk POS tagging, dependency parsing, dan NER basic.
nlp = spacy.load("en_core_web_sm")


# ============================================================
# 1. GRAMMAR PARSING WITH NLTK
# ============================================================

print("\n===================================================")
print("1. GRAMMAR PARSING WITH NLTK")
print("===================================================")

# CFG = Context-Free Grammar.
# Di sini kita mendefinisikan aturan grammar secara manual.
#
# S  = Sentence
# NP = Noun Phrase
# VP = Verb Phrase
# PP = Prepositional Phrase
# Det = Determiner
# Adj = Adjective
# N = Noun
# V = Verb
# P = Preposition
grammar = CFG.fromstring("""
S -> NP VP

NP -> Det N
NP -> Det Adj N
NP -> N

VP -> V
VP -> V NP
VP -> V PP
VP -> V NP PP

PP -> P NP

Det -> 'the' | 'a'
Adj -> 'small' | 'big' | 'red'
N -> 'cat' | 'dog' | 'fish' | 'park' | 'boy' | 'telescope'
V -> 'eats' | 'sees' | 'walks'
P -> 'in' | 'with'
""")

print("\nGrammar Rules:")
print(grammar)

# Beberapa kalimat contoh untuk dianalisis.
# Catatan: kata-kata di kalimat harus tersedia di grammar rule.
sentences_for_parsing = [
    "the cat eats fish",
    "the small dog walks in the park",
    "the boy sees the dog with a telescope"
]

# ChartParser digunakan untuk menghasilkan parse tree berdasarkan grammar.
chart_parser = ChartParser(grammar)

for sentence in sentences_for_parsing:
    print("\n---------------------------------------------------")
    print("Sentence:", sentence)

    # Tokenisasi sederhana menggunakan split.
    # Karena grammar kita semua lowercase, sentence juga dibuat lowercase.
    tokens = sentence.lower().split()
    print("Tokens:", tokens)

    # Parse sentence berdasarkan grammar.
    parse_trees = list(chart_parser.parse(tokens))

    if len(parse_trees) == 0:
        print("No parse tree found.")
    else:
        for i, tree in enumerate(parse_trees, start=1):
            print(f"\nParse Tree {i}:")

            # Print struktur tree dalam bentuk bracket notation.
            print(tree)

            # Print tree dalam bentuk visual text.
            tree.pretty_print()

            # Height menunjukkan kedalaman struktur tree.
            print("Tree Height:", tree.height())

            # Leaves adalah kata-kata asli di bagian paling bawah tree.
            print("Tree Leaves:", tree.leaves())


# ============================================================
# 2. EXTRACT PHRASES FROM PARSE TREE
# ============================================================

print("\n===================================================")
print("2. EXTRACT PHRASES FROM PARSE TREE")
print("===================================================")

def extract_phrases(tree, phrase_label):
    """
    Fungsi untuk mengambil phrase tertentu dari parse tree.

    Contoh phrase_label:
    - "NP" untuk Noun Phrase
    - "VP" untuk Verb Phrase
    - "PP" untuk Prepositional Phrase
    """

    phrases = []

    # Loop semua subtree dalam parse tree.
    for subtree in tree.subtrees():

        # Jika label subtree sama dengan label yang dicari,
        # ambil semua kata di subtree tersebut.
        if subtree.label() == phrase_label:
            phrase = " ".join(subtree.leaves())
            phrases.append(phrase)

    return phrases


sentence = "the small dog walks in the park"
tokens = sentence.lower().split()
trees = list(chart_parser.parse(tokens))

if trees:
    tree = trees[0]

    noun_phrases = extract_phrases(tree, "NP")
    verb_phrases = extract_phrases(tree, "VP")
    prepositional_phrases = extract_phrases(tree, "PP")

    print("Sentence:", sentence)
    print("Noun Phrases:", noun_phrases)
    print("Verb Phrases:", verb_phrases)
    print("Prepositional Phrases:", prepositional_phrases)


# ============================================================
# 3. DEPENDENCY PARSING WITH SPACY
# ============================================================

print("\n===================================================")
print("3. DEPENDENCY PARSING WITH SPACY")
print("===================================================")

# Dependency parsing bertujuan memahami hubungan antar kata.
# Contoh:
# "cat" adalah subject dari "eats"
# "fish" adalah object dari "eats"
dependency_sentences = [
    "The cat eats fish.",
    "The small dog walks in the park.",
    "The boy sees the dog with a telescope.",
    "Apple opened a new office in Jakarta on Monday."
]

for sentence in dependency_sentences:
    print("\n---------------------------------------------------")
    print("Sentence:", sentence)

    # Proses sentence menggunakan pipeline SpaCy.
    doc = nlp(sentence)

    dependency_data = []

    for token in doc:
        dependency_data.append({
            # Token asli
            "Token": token.text,

            # Lemma = bentuk dasar kata
            "Lemma": token.lemma_,

            # POS = kategori kata umum, misalnya NOUN, VERB, PROPN
            "POS": token.pos_,

            # Tag = POS yang lebih detail, misalnya NN, VBD, DT
            "Tag": token.tag_,

            # Dependency = relasi token terhadap head-nya
            "Dependency": token.dep_,

            # Head = kata utama tempat token ini bergantung
            "Head": token.head.text,

            # Children = kata-kata yang bergantung pada token ini
            "Children": [child.text for child in token.children]
        })

    # Tampilkan hasil dalam bentuk tabel agar mudah dibaca.
    df_dependency = pd.DataFrame(dependency_data)
    print(df_dependency)


# ============================================================
# 4. EXPLAIN DEPENDENCY LABELS
# ============================================================

print("\n===================================================")
print("4. EXPLAIN DEPENDENCY LABELS")
print("===================================================")

sentence = "Apple opened a new office in Jakarta on Monday."
doc = nlp(sentence)

# SpaCy menyediakan spacy.explain() untuk menjelaskan arti label dependency.
for token in doc:
    print(
        f"{token.text:10} | "
        f"dep: {token.dep_:10} | "
        f"meaning: {spacy.explain(token.dep_)}"
    )


# ============================================================
# 5. SIMPLE SUBJECT-VERB-OBJECT EXTRACTION
# ============================================================

print("\n===================================================")
print("5. SIMPLE SUBJECT-VERB-OBJECT EXTRACTION")
print("===================================================")

def extract_svo(sentence):
    """
    Fungsi sederhana untuk mengambil:
    - Subject
    - Root Verb
    - Object

    Catatan:
    Ini pendekatan sederhana, bukan semantic parser sempurna.
    """

    doc = nlp(sentence)

    subject = []
    verb = []
    object_ = []

    for token in doc:

        # nsubj = nominal subject
        # nsubjpass = passive nominal subject
        if token.dep_ in ["nsubj", "nsubjpass"]:
            subject.append(token.text)

        # ROOT biasanya main verb atau kata inti dalam kalimat.
        if token.dep_ == "ROOT":
            verb.append(token.text)

        # dobj = direct object
        # pobj = object of preposition
        # attr = attribute
        if token.dep_ in ["dobj", "pobj", "attr"]:
            object_.append(token.text)

    return {
        "sentence": sentence,
        "subject": subject,
        "verb": verb,
        "object": object_
    }


svo_examples = [
    "The cat eats fish.",
    "Apple opened a new office in Jakarta.",
    "Elon Musk founded SpaceX."
]

for sentence in svo_examples:
    result = extract_svo(sentence)
    print(result)


# ============================================================
# 6. NAMED ENTITY RECOGNITION WITH SPACY
# ============================================================

print("\n===================================================")
print("6. NAMED ENTITY RECOGNITION WITH SPACY")
print("===================================================")

# NER bertujuan mendeteksi entity penting seperti:
# PERSON, ORG, GPE, DATE, MONEY, dll.
ner_texts = [
    "Elon Musk founded SpaceX in California in 2002.",
    "Apple opened a new office in Jakarta on Monday.",
    "Microsoft invested $10 billion in OpenAI.",
    "Barack Obama was born in Hawaii."
]

for text in ner_texts:
    print("\n---------------------------------------------------")
    print("Text:", text)

    doc = nlp(text)

    if len(doc.ents) == 0:
        print("No entities found.")
    else:
        for ent in doc.ents:
            print(
                f"Entity: {ent.text:20} | "
                f"Label: {ent.label_:10} | "
                f"Meaning: {spacy.explain(ent.label_)}"
            )


# ============================================================
# 7. ENTITY COUNTING
# ============================================================

print("\n===================================================")
print("7. ENTITY COUNTING")
print("===================================================")

large_text = """
Elon Musk founded SpaceX in California in 2002.
Apple opened a new office in Jakarta on Monday.
Microsoft invested $10 billion in OpenAI.
Barack Obama was born in Hawaii.
Google operates offices in Singapore and Indonesia.
"""

doc = nlp(large_text)

entity_counts = {}

# Hitung jumlah kemunculan tiap label entity.
for ent in doc.ents:
    entity_counts[ent.label_] = entity_counts.get(ent.label_, 0) + 1

print("Entity Counts:")

for label, count in entity_counts.items():
    print(f"{label}: {count} ({spacy.explain(label)})")


# ============================================================
# 8. CONVERT NER RESULT TO DATAFRAME
# ============================================================

print("\n===================================================")
print("8. NER RESULT AS DATAFRAME")
print("===================================================")

entity_data = []

for ent in doc.ents:
    entity_data.append({
        "Entity": ent.text,
        "Label": ent.label_,
        "Explanation": spacy.explain(ent.label_),

        # Posisi awal entity dalam teks
        "Start Char": ent.start_char,

        # Posisi akhir entity dalam teks
        "End Char": ent.end_char
    })

df_entities = pd.DataFrame(entity_data)
print(df_entities)


# ============================================================
# 9. COMBINED ANALYSIS FUNCTION
# ============================================================

def analyze_text(text):
    """
    Fungsi gabungan untuk menganalisis satu teks dengan:
    1. POS Tagging
    2. Dependency Parsing
    3. Named Entity Recognition
    4. Noun Chunk Extraction
    """

    doc = nlp(text)

    print("\n===================================================")
    print("COMBINED TEXT ANALYSIS")
    print("===================================================")
    print("Text:", text)

    # --------------------------------------------------------
    # A. Token, POS, Dependency
    # --------------------------------------------------------
    print("\nA. Token, POS, Dependency")
    print("-----------------------------------------------")

    for token in doc:
        print(
            f"{token.text:12} | "
            f"POS: {token.pos_:8} | "
            f"DEP: {token.dep_:10} | "
            f"HEAD: {token.head.text}"
        )

    # --------------------------------------------------------
    # B. Named Entities
    # --------------------------------------------------------
    print("\nB. Named Entities")
    print("-----------------------------------------------")

    if len(doc.ents) == 0:
        print("No entities found.")
    else:
        for ent in doc.ents:
            print(
                f"{ent.text:20} | "
                f"{ent.label_:10} | "
                f"{spacy.explain(ent.label_)}"
            )

    # --------------------------------------------------------
    # C. Noun Chunks
    # --------------------------------------------------------
    print("\nC. Noun Chunks")
    print("-----------------------------------------------")

    # Noun chunk adalah phrase berbasis noun.
    # Contoh: "a new office", "Jakarta"
    for chunk in doc.noun_chunks:
        print(
            f"{chunk.text:25} | "
            f"Root: {chunk.root.text:10} | "
            f"Dependency: {chunk.root.dep_}"
        )


# ============================================================
# 10. RUN COMBINED ANALYSIS
# ============================================================

print("\n===================================================")
print("10. RUN COMBINED ANALYSIS")
print("===================================================")

sample_text = "Apple opened a new office in Jakarta on Monday."

analyze_text(sample_text)


# ============================================================
# 11. USER INPUT ANALYSIS
# ============================================================

print("\n===================================================")
print("11. USER INPUT ANALYSIS")
print("===================================================")

# User bisa memasukkan kalimat sendiri untuk dianalisis.
user_text = input("Input an English sentence: ")

if user_text.strip():
    analyze_text(user_text)
else:
    print("No input provided.")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!



1. GRAMMAR PARSING WITH NLTK

Grammar Rules:
Grammar with 25 productions (start state = S)
    S -> NP VP
    NP -> Det N
    NP -> Det Adj N
    NP -> N
    VP -> V
    VP -> V NP
    VP -> V PP
    VP -> V NP PP
    PP -> P NP
    Det -> 'the'
    Det -> 'a'
    Adj -> 'small'
    Adj -> 'big'
    Adj -> 'red'
    N -> 'cat'
    N -> 'dog'
    N -> 'fish'
    N -> 'park'
    N -> 'boy'
    N -> 'telescope'
    V -> 'eats'
    V -> 'sees'
    V -> 'walks'
    P -> 'in'
    P -> 'with'

---------------------------------------------------
Sentence: the cat eats fish
Tokens: ['the', 'cat', 'eats', 'fish']

Parse Tree 1:
(S (NP (Det the) (N cat)) (VP (V eats) (NP (N fish))))
             S               
      _______|________        
     |                VP     
     |            ____|___    
     NP          |        NP 
  ___|___        |        |   
Det      N       V        N  
 |       |       |        |   
the     cat     eats     fish

Tree Height: 5
Tree Leaves: ['the', 'cat', 